In [ ]:
import os
import json
import pandas as pd
import numpy as np
import scipy as sp
import pylab as plt
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem import WordNetLemmatizer

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
plt.style.use("../src/mpl_style.txt")

In [ ]:
import jupyter_black
jupyter_black.load()

In [ ]:
BASELINE_NAME = "baseline_2025-06-26"
FREQS_PATH = os.path.join("../data", BASELINE_NAME, "freqs")

In [ ]:
df1 = pd.read_json(os.path.join("../data", BASELINE_NAME, "formatted", "2000.json"))
len(df1)

In [ ]:
df2 = pd.read_json(os.path.join("../data", BASELINE_NAME, "formatted", "2001.json"))
len(df2)

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
len(df)

In [ ]:
sections = ["introduction", "methods", "results", "discussion"]
df = df[df["sections"].apply(lambda x: set(x.keys()) == set(sections))]
len(df)

In [ ]:
df["full"] = [""] * len(df)

In [ ]:
for sec in sections:
    df[sec] = list(map((lambda x: x[sec]), df["sections"]))
    df["full"] = df["full"] + df[sec]

#### Vectorize abstracts

In [ ]:
vectorizer = CountVectorizer(binary=True, min_df=1e-6)
X = vectorizer.fit_transform(df.abstract.values)

In [ ]:
X.shape

In [ ]:
words = vectorizer.get_feature_names_out()
words[2000:2010]

In [ ]:
months = np.arange(1, 13)
months

In [ ]:
years = np.arange(2000, 2002)
years

In [ ]:
counts = np.zeros((words.size, months.size * years.size))
totals = np.zeros(months.size * years.size)

In [ ]:
print(f"{df["date"].iloc[0].month}-{df["date"].iloc[0].year}")

In [ ]:
for j, year in enumerate(years):
    for i, month in enumerate(months):
        ind = df["date"].apply(lambda x: (x.month == month) and (x.year == year)).values
        # count how many times a word appears in each month
        counts[:, 12 * j + i] = np.array(np.sum(X[ind, :], axis=0)).ravel()
        # count papers per month
        totals[12 * j + i] = np.sum(ind)

In [ ]:
totals

In [ ]:
months_years = [f"{m}-{y}" for y in years for m in months]

In [ ]:
# df with each row corresponding to the counts for one word in each month
count_df = pd.DataFrame(
    dict(zip(months_years, list(counts.astype(int).T))), index=words
)

In [ ]:
count_df

group the counts of different word versions together

TODO: think about doing americanization and lemmatization at the same time

In [ ]:
lemmas = utils.get_lemma_dict(words)

In [ ]:
len(lemmas)

In [ ]:
list(lemmas.items())[20]

In [ ]:
count_df.loc["abnormalities"]

In [ ]:
count_df.loc["abnormality"]

In [ ]:
with open("../src/british_spellings.json") as f:
    british_spell = json.load(f)

In [ ]:
count_df = count_df.rename(index=british_spell)
count_df = count_df.groupby(count_df.index).sum()
len(count_df)

In [ ]:
count_df = count_df.rename(index=lemmas)
count_df = count_df.groupby(count_df.index).sum()
len(count_df)

In [ ]:
count_df.loc["abnormality"]

merging dataframes along the rows

In [ ]:
words1 = ["apple", "banana", "berry"]
words2 = ["apples", "banana", "berry", "peach", "strawberry"]
words3 = ["apple", "strawberry"]
days1 = ["monday", "tuesday"]
days2 = ["wednesday", "thursday"]
days3 = ["friday", "saturday"]
counts1 = np.array([[1, 2], [2, 1], [3, 3]])
counts2 = np.array([[2, 2], [2, 2], [1, 1], [3, 3], [1, 1]])
counts3 = np.array([[1, 3], [3, 1]])

In [ ]:
df1 = pd.DataFrame(dict(zip(days1, list(counts1.astype(int).T))), index=words1)
df1

In [ ]:
df2 = pd.DataFrame(dict(zip(days2, list(counts2.astype(int).T))), index=words2)
df2

In [ ]:
df3 = pd.DataFrame(dict(zip(days3, list(counts3.astype(int).T))), index=words3)
df3

In [ ]:
# collect dfs in list and merge only once with the full list to optimize performance
df123 = pd.concat([df1, df2, df3], axis=1).fillna(0)
df123

In [ ]:
# regroup similar words into one index (non existing indexes are ignored)
df123 = df123.rename(index={"strawberry": "berry", "apples": "apple", "aeon": "eon"})
df123.groupby(df123.index).sum()

#### Compute excess

In [ ]:
freqs = (counts + 1) / (totals + 1)

In [ ]:
freqs.shape

In [ ]:
alphabet = "abcdefghijklmnopqrstuvwxyz"
allowedWords = np.array([np.all([s in alphabet for s in w]) for w in words])
allowedWords &= np.array([len(w) >= 4 for w in words])

In [ ]:
allowedWords

In [ ]:
freqs_df = pd.DataFrame(dict(zip(list(months), list(freqs.T))), index=words)

In [ ]:
freqs_df = freqs_df.rename(index=british_spell).rename(index=lemmas)
freqs_df = freqs_df.groupby(freqs_df.index).sum()
freqs_df

### replicate frequency plot from paper

In [ ]:
selection = [
    "delves",
    "crucial",
    "potential",
    "these",
    "significant",
    "important",
    "pandemic",
    "ebola",
    "convolutional",
]
years = np.arange(2010, 2026)

In [ ]:
freqs_yearly = dict(zip(selection, np.zeros((len(selection), years.size))))
freqs_monthly = dict(zip(selection, np.zeros((len(selection), years.size * 12 - 6))))

In [ ]:
x_months = np.arange(2010, 2025.5, 1 / 12)

In [ ]:
for i, year in enumerate(years):
    if year == 2025:
        drop = ["word", "7", "8", "9", "10", "11", "12"]
    else:
        drop = "word"

    freqs_df = pd.read_csv(os.path.join(FREQS_PATH, f"freq_df_{year}.csv.gz"))

    for w in selection:
        if w in freqs_df.word.values:
            freq = freqs_df[freqs_df["word"] == w].drop(labels=drop, axis=1)

            freqs_yearly[w][i] = freq.mean(axis=1).values[0]

            if year == 2025:
                freqs_monthly[w][i * 12 :] = freq.values
            else:
                freqs_monthly[w][i * 12 : (i + 1) * 12] = freq.values

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(3.5, 3), layout="constrained")

for i, word in enumerate(selection):
    freq = freqs_yearly[word]
    axs.flat[i].plot(years, freq, ".-", clip_on=False)
    axs.flat[i].set_title(word)
    if i < 6:
        axs.flat[i].set_xticks([2012, 2016, 2020, 2024])
        axs.flat[i].set_xticklabels([])
    else:
        axs.flat[i].set_xticks([2012, 2016, 2020, 2024])
        axs.flat[i].set_xticklabels([2012, 2016, 2020, 2024], rotation=60)
    if i in [0, 3, 6]:
        axs.flat[i].set_ylabel("Frequency")
    if i < 6:
        proj = freq[-4] + np.maximum(0, (freq[-4] - freq[-5])) * 2
        axs.flat[i].plot([2022, 2024], [freq[-4], proj], "k-", zorder=0)

    axs.flat[i].spines[["right", "top"]].set_visible(False)

fig.align_ylabels()

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(7, 3), layout="constrained")

for i, word in enumerate(selection):
    freq = freqs_yearly[word]
    freq_m = freqs_monthly[word]

    axs.flat[i].plot(years + 0.5, freq, ".-", clip_on=False)
    axs.flat[i].plot(x_months, freq_m, "k-", clip_on=False, alpha=0.2)
    axs.flat[i].set_title(word)
    if i < 6:
        axs.flat[i].set_xticks([2012.5, 2016.5, 2020.5, 2024.5])
        axs.flat[i].set_xticklabels([])
    else:
        axs.flat[i].set_xticks([2012.5, 2016.5, 2020.5, 2024.5])
        axs.flat[i].set_xticklabels([2012, 2016, 2020, 2024], rotation=60)
    if i in [0, 3, 6]:
        axs.flat[i].set_ylabel("Frequency")
    if i < 6:
        proj = freq[-4] + np.maximum(0, (freq[-4] - freq[-5])) * 2
        axs.flat[i].plot([2022, 2024], [freq[-4], proj], "k-", zorder=0)

    axs.flat[i].spines[["right", "top"]].set_visible(False)

fig.align_ylabels()